# Кластеризация точек по ассортиментному профилю

Главный ноутбук пайплайна. Вопрос: чем одна точка "похожа" или "непохожа" на другую по тому, **какие товарные
категории делают её выручку**, а не по общим цифрам продаж.

В коде опробовано подряд несколько способов посчитать это "похожа/непохожа", от простого к сложному - здесь
они разложены на варианты A/B/C/D, чтобы был виден ход рассуждения:

- **A.** Точки ранжируются отдельно по каждой категории - по абсолютной (логарифмированной) выручке в этой категории.
- **B.** Точки ранжируются отдельно по каждой категории - по доле точки в выручке **всей сети** по этой категории
  (кто из точек - основной "вклад" в продажи категории по сети).
- **C.** Точки ранжируются отдельно по каждой категории - по доле категории в выручке **самой точки**
  (ассортиментный профиль/микс точки - на что она сама больше "заточена").
- **D.** Финальный вариант: не по одной категории за раз, а совместная кластеризация по вектору из всех
  категорий сразу (профиль C, но без разбивки на категории по отдельности) - это то, что дальше сопоставляется
  с брендом сети, локацией и форматом выкладки точки.

Подключение к БД и товарные справочники (категории, объединение мета-категорий) в портфолио не включены.

## Импорт библиотек

In [ ]:
import pandas as pd
import numpy as np
import sqlalchemy

import seaborn as sns
import matplotlib as mpl
import matplotlib.pyplot as plt
%matplotlib inline
import scipy.cluster.hierarchy as shc
from sklearn.cluster import AgglomerativeClustering
from sklearn.preprocessing import StandardScaler
from sklearn.manifold import TSNE

import gc

pd.options.display.max_columns = 100

## Настройка параметров

In [ ]:
# Периоды для выгрузки чеков (в исходном рабочем ноутбуке сокращены до нескольких дней для быстрого тестового прогона)
lb_calc_period_1 = pd.to_datetime('20220201')
rb_calc_period_1 = pd.to_datetime('20220202')

lb_calc_period_2 = pd.to_datetime('20220203')
rb_calc_period_2 = pd.to_datetime('20220204')

lb_calc_period_3 = pd.to_datetime('20220205')
rb_calc_period_3 = pd.to_datetime('20220206')

OPTICS_TRADE_POINT_TYPE_ID = '4'

# товарные группы, не считающиеся товарной выручкой в узком смысле - исключаются из расчёта
EXCLUDED_AP_GROUPS = ['DD', 'N']

EXCLUDED_COUNTERPARTY_GROUP_ID = 8538
EXCLUDED_MANUFACTURER_FILE = 'исключённый_производитель_позиции.xlsx'  # см. 04_..._fassovka.ipynb

# число ранжированных кластеров для вариантов A и B/C соответственно
N_CLUSTERS_ABSOLUTE = 15
N_CLUSTERS_SHARE = 5

# финальный (вариант D) - совместная кластеризация по полному вектору категорий
N_CLUSTERS_JOINT = 18

## Функции

In [ ]:
def get_db_engine(name: str):
    """Заглушка вместо реального подключения к корпоративному DWH (SQL Server, ODBC) - см. 01_get_optics.ipynb."""
    raise NotImplementedError("Подключение к реальной БД в портфолио-версии не используется")


def batch_load_sql_dwh(query: str) -> pd.DataFrame:
    """Чанкованная загрузка результата SQL-запроса из DWH."""
    CHUNKSIZE = 200_000
    engine = get_db_engine('dwh')
    conn = engine.connect().execution_options(stream_results=True)
    chunks = []
    for chunk_dataframe in pd.read_sql(query, conn, chunksize=CHUNKSIZE):
        chunks.append(chunk_dataframe)
    conn.close()
    return pd.concat(chunks, ignore_index=True)

В исходном рабочем ноутбуке один и тот же цикл ранжированной кластеризации "по каждой категории отдельно"
был скопирован 3 раза почти без изменений (варианты A, B, C ниже отличались только тем, какую именно колонку
подавали на вход). Здесь это вынесено в одну функцию.

In [ ]:
def rank_clusters_per_column(df_wide: pd.DataFrame, n_clusters: int) -> pd.DataFrame:
    """
    Кластеризует каждую колонку датафрейма независимо (1D-кластеризация Ward), затем перенумеровывает
    кластеры внутри каждой колонки так, чтобы кластер 0 всегда был с самым высоким средним значением,
    а последний - с самым низким. Так кластерные id сопоставимы между разными категориями/колонками.
    """
    result = {}
    for i in range(df_wide.shape[1]):
        column = pd.DataFrame(df_wide.iloc[:, i])

        cluster = AgglomerativeClustering(n_clusters=n_clusters, affinity='euclidean', linkage='ward')
        labels = cluster.fit_predict(column)

        tmp = column.assign(ID_groups=labels)
        cluster_order = tmp.groupby('ID_groups').agg('mean').sort_values(by=tmp.columns[0], ascending=False).reset_index()
        rank_by_old_id = dict(zip(cluster_order.ID_groups, cluster_order.index.values))

        tmp = tmp.replace({'ID_groups': rank_by_old_id})
        col_name = tmp.columns[0]
        result[i] = tmp.drop([col_name], axis=1).rename({'ID_groups': col_name}, axis=1)

    result_df = pd.concat(result, axis=1)
    result_df.columns = result_df.columns.droplevel()
    return result_df


def attach_trade_point_info(df_wide: pd.DataFrame) -> pd.DataFrame:
    """Добавляет к результату кластеризации код точки, бренд сети - общий финальный шаг для всех вариантов."""
    out = df_wide.reset_index()
    out = out.merge(trade_points[['STORE_LOCATION_ID', 'TradePointCode',
                                   'PRODUCT_ATTR_NM', 'STORE_LOCATION_LVL_NM1', 'STORE_LOCATION_NM']],
                     left_on='TradePointId', right_on='STORE_LOCATION_ID', how='left').drop(['STORE_LOCATION_ID'], axis=1)
    out = out.merge(trade_points_brands, how='left', on='TradePointCode')
    out = out.drop_duplicates('TradePointCode', keep='first')
    return out.drop(['PRODUCT_ATTR_NM', 'TradePointId'], axis=1)

## Алгоритм выполнения

### Отбор точек и товарных категорий

In [ ]:
trade_points = pd.read_sql(f"""
    SELECT t1.STORE_LOCATION_ID,
        SUBSTRING(t1.STORE_LOCATION_NM,1,6) as TradePointCode,
        t1.STORE_LOCATION_LVL_ID3,
        t2.PRODUCT_ATTR_NM,
        t1.STORE_LOCATION_LVL_NM1,
        t1.CLOSE_DTTM,
        t1.OPEN_DTTM,
        tp.TradeSquare,
        t1.STORE_LOCATION_NM
    FROM dwh.IN_STORE_LOCATION_NEW t1
    INNER JOIN dwh.IN_CLUSTER_BASKET t2 ON (t1.STORE_LOCATION_LVL_ID3 = t2.STORE_LOCATION_LVL_ID3)
    INNER JOIN dwh.TradePoints tp ON tp.TradePointId = t1.STORE_LOCATION_ID
    WHERE tp.TradePointTypeId != '{OPTICS_TRADE_POINT_TYPE_ID}'
    """, con=get_db_engine('sas_ia'))
trade_points['CLOSE_DTTM'] = pd.to_datetime(trade_points['CLOSE_DTTM'])

trade_points = trade_points.loc[(trade_points.CLOSE_DTTM > rb_calc_period_3)]
trade_points = trade_points.astype({"CLOSE_DTTM": 'datetime64[ns]', "OPEN_DTTM": 'datetime64[ns]'})
trade_points = trade_points.assign(time_dur_to_now=lambda x: (rb_calc_period_3 - x.OPEN_DTTM) / np.timedelta64(1, 'M'))
trade_points = trade_points.query('time_dur_to_now >= 6')

trade_points_brands = pd.read_sql("""
SELECT t2.BrandName, t1.TradePointCode
FROM dwh.TradePoints t1
INNER JOIN dwh.TradePointBrands t2 ON t1.BrandId = t2.BrandId
""", con=get_db_engine('sas_ia'))

Товарные категории - тот же справочник, что и в предыдущих ноутбуках. Дополнительно пара смежных категорий
объединяется в один укрупнённый кластер - для целей ассортиментного анализа разница между ними не так важна,
как для ценового.

In [ ]:
ap_cat_filtr = pd.read_excel('Категории для кластеризации.xlsx', 'Категории для кластеризации')
ap_cat_filtr = ap_cat_filtr.loc[ap_cat_filtr['Категория для кластеризации'] != 'удалить из данных']
ap_cat_filtr = ap_cat_filtr.loc[ap_cat_filtr['для категорий и ценовой кластеризации'] != 'удалить из данных']
groups_to_keep = ap_cat_filtr['2-Группа']

ap_cat_filtr = ap_cat_filtr.rename({'2-Группа': 'Groups', 'Категория для кластеризации': 'Clust_groups'}, axis=1)

### Чеки

Те же бизнес-фильтры, что и в предыдущих ноутбуках (см. `02`-`04`), плюс исключение товарной группы `N` и
полное исключение одного производителя (та же логика, что в `04_..._fassovka.ipynb`).

Коллега как-то подсказал по этому фильтру идею на будущее: по аналогии с исключением АптекаРу/Здравсити стоит
добавить и другие похожие маркетплейсы-агрегаторы - на момент этого расчёта их ещё не добавили.

In [ ]:
query_filt = """
SELECT
    fc.ChequeGUId,
    TradePointId,
    fc.ProductId,
    ap.apCode,
    Date,
    Time,
    Sum_Fact,
    Amount,
    Sum_Purch,
    Sum_Price,
    rowNumber as ChequePos,
    ChequeHeadersId,
    voda.Groups
FROM dbo.Fact_cheque fc
  JOIN dbo.InternetOrders io ON fc.InternetOrderId = io.InternetOrderId
  JOIN dbo.InternetOrderTypes iot ON fc.InternetOrderTypeId = iot.InternetOrderTypeId
  JOIN dbo.Parts p ON fc.PartId = p.PartId
  JOIN dbo.Products prod ON prod.ProductId = fc.ProductId
  JOIN dwh.dimAP ap ON ap.apID = prod.apId
  JOIN dbo.v_OLAP_dimAP voda ON voda.apCode = ap.apCode
  JOIN dwh.DicCounterparty d ON fc.DicCustomerId = d.DicCounterpartyId
  JOIN dwh.Counterparties c ON d.CounterpartyId = c.CounterpartyId
  JOIN dwh.CounterpartyGroups g ON c.GroupId = g.GroupId
WHERE fc.IsInternalId=0
  AND (fc.SaleTypeId=1 OR fc.SaleTypeId = 3 OR fc.SaleTypeId = 4 OR fc.SaleTypeId = 5)
  AND io.InternetOrderSourceId NOT IN (6, 38)
  AND datediff(month, fc.[Date],p.ExpireDate)>=6
  AND voda.Class!='Нетоварные позиции'
  AND TradePointId IN {}
  AND ((Date BETWEEN {} AND {}) OR (DATE BETWEEN {} AND {}) OR (DATE BETWEEN {} AND {}))
  AND voda.Groups!='UN'
  AND g.GroupID != %d
""" % EXCLUDED_COUNTERPARTY_GROUP_ID

df = batch_load_sql_dwh(query_filt.format(tuple(trade_points.STORE_LOCATION_ID),
          "'" + str(lb_calc_period_1.date()) + "'", "'" + str(rb_calc_period_1.date()) + "'",
          "'" + str(lb_calc_period_2.date()) + "'", "'" + str(rb_calc_period_2.date()) + "'",
          "'" + str(lb_calc_period_3.date()) + "'", "'" + str(rb_calc_period_3.date()) + "'"))

df = df.query('Groups in @groups_to_keep')
df = df.merge(ap_cat_filtr[['Groups', 'Clust_groups']], on='Groups', how='left')
df['Clust_groups'][df['Clust_groups'].isin(['Сердечно-сосудистая система', 'Кровь и кроветворение'])] = 'Сердце и кровь'

In [ ]:
excluded_groups = batch_load_sql_dwh(f"""SELECT apCode FROM dwh.dimAP da
JOIN dwh.dimAPGroups da2 ON da.apGroupID = da2.apGroupID
WHERE apGroupCode IN {tuple(EXCLUDED_AP_GROUPS)}
""")
df = df[~df.apCode.isin(list(excluded_groups.apCode))]

excluded_manufacturer = pd.read_excel(EXCLUDED_MANUFACTURER_FILE)
excluded_manufacturer = excluded_manufacturer.rename({'Код АП': 'apCode'}, axis=1).astype({'apCode': 'object'})
df = df[~df.apCode.isin(list(map(str, excluded_manufacturer.apCode)))]

### Вариант A: абсолютная выручка категории (логарифм), ранжированные кластеры

Идея: для каждой категории отдельно смотрим, сколько точка на ней зарабатывает в абсолютных деньгах
(логарифм, чтобы сгладить разброс), и делим точки на 15 ранжированных сегментов по каждой категории отдельно.

In [ ]:
category_revenue = df.groupby(['TradePointId', 'Clust_groups'], as_index=False).agg({'Sum_Price': 'sum'})
category_revenue.rename({'Sum_Price': 'Revenue'}, axis=1, inplace=True)
category_revenue = category_revenue.assign(Revenue_ln=np.log(category_revenue.Revenue))

category_revenue_ln_wide = category_revenue[['TradePointId', 'Clust_groups', 'Revenue_ln']]\
    .pivot(index='TradePointId', columns='Clust_groups', values='Revenue_ln').fillna(0)

standard = StandardScaler(with_mean=True, with_std=True)
category_revenue_ln_wide_scaled = pd.DataFrame(
    standard.fit_transform(category_revenue_ln_wide),
    index=category_revenue_ln_wide.index, columns=category_revenue_ln_wide.columns,
)

variant_a = rank_clusters_per_column(category_revenue_ln_wide_scaled, N_CLUSTERS_ABSOLUTE)
variant_a = attach_trade_point_info(variant_a)
# variant_a.to_excel("each_group_ln_sum_clustering.xlsx", index=False)
variant_a

### Вариант B: доля точки в выручке сети по категории

Идея: для каждой категории отдельно смотрим, какую долю выручки сети по этой категории делает конкретная
точка (не в своих продажах, а во всей сети) - то есть кто "тянет" продажи категории по сети.

In [ ]:
category_revenue_by_network = df.groupby('Clust_groups', as_index=False).agg({'Sum_Price': 'sum'})
category_revenue_by_network.rename({'Sum_Price': 'NetworkRevenue'}, axis=1, inplace=True)

share_of_network = category_revenue.merge(category_revenue_by_network, on='Clust_groups')
share_of_network = share_of_network.assign(ratio_group=lambda x: x.Revenue / x.NetworkRevenue)

share_of_network_wide = share_of_network[['TradePointId', 'Clust_groups', 'ratio_group']]\
    .pivot(index='TradePointId', columns='Clust_groups', values='ratio_group').fillna(0)

variant_b = rank_clusters_per_column(share_of_network_wide, N_CLUSTERS_SHARE)
variant_b = attach_trade_point_info(variant_b)
# variant_b.to_excel("each_group_relative_to_all_tt_ratio_clustering.xlsx", index=False)
variant_b

### Вариант C: доля категории в собственной выручке точки (ассортиментный профиль)

Идея: для каждой категории отдельно смотрим, какую долю выручки **самой точки** делает эта категория -
это про то, на что точка сама "заточена" ассортиментно, независимо от её общего размера.

In [ ]:
store_total_revenue = df.groupby('TradePointId', as_index=False).agg({'Sum_Price': 'sum'})
store_total_revenue.rename({'Sum_Price': 'Revenue'}, axis=1, inplace=True)

assortment_mix = category_revenue.merge(store_total_revenue, on='TradePointId', suffixes=('', '_total'))
assortment_mix = assortment_mix.assign(ratio_group=lambda x: x.Revenue / x.Revenue_total)

assortment_mix_wide = assortment_mix[['TradePointId', 'Clust_groups', 'ratio_group']]\
    .pivot(index='TradePointId', columns='Clust_groups', values='ratio_group').fillna(0)

variant_c = rank_clusters_per_column(assortment_mix_wide, N_CLUSTERS_SHARE)
variant_c = attach_trade_point_info(variant_c)
# variant_c.to_excel("each_group_ratio_clustering.xlsx", index=False)
variant_c

### Вариант D (финальный): совместная кластеризация по полному ассортиментному профилю

В отличие от A/B/C, здесь не кластеризуем каждую категорию отдельно, а берём весь вектор долей категорий
(из варианта C, `assortment_mix_wide`) сразу и кластеризуем точки как единое целое - это даёт не 18 наборов
рангов "по каждой категории", а один итоговый ассортиментный кластер на точку. Число кластеров (18) выбрано
по дендрограмме и "elbow"-графику (вторая производная расстояний), как и в предыдущих ноутбуках.

In [ ]:
assortment_mix_scaled = pd.DataFrame(
    standard.fit_transform(assortment_mix_wide),
    index=assortment_mix_wide.index, columns=assortment_mix_wide.columns,
)

mpl.rc_file_defaults()
dend = shc.dendrogram(shc.linkage(assortment_mix_scaled, method='ward'),
                       truncate_mode='lastp', p=N_CLUSTERS_JOINT, show_leaf_counts=True)

In [ ]:
cluster = AgglomerativeClustering(n_clusters=N_CLUSTERS_JOINT, affinity='euclidean', linkage='ward')
clst_assortment = cluster.fit_predict(assortment_mix_scaled)

pd.Series(clst_assortment, name='ClusterId').value_counts()

In [ ]:
fig = plt.figure(figsize=(15, 9))
Z = shc.linkage(assortment_mix_scaled, method='ward')
last = Z[-68:, 2]
last_rev = last[::-1]
idxs = np.arange(1, len(last) + 1)
plt.plot(idxs, last_rev)

acceleration = np.diff(last, 2)
acceleration_rev = acceleration[::-1]
plt.plot(idxs[:-2] + 1, acceleration_rev)

plt.axvline(x=N_CLUSTERS_JOINT, ymax=0.6, color='r', ls=':', lw=2)
plt.show()

In [ ]:
tsne_coords = TSNE(random_state=2023).fit_transform(assortment_mix_scaled)
tsne_coords = pd.DataFrame(tsne_coords).rename({0: 'x', 1: 'y'}, axis=1)
tsne_coords = tsne_coords.assign(ID_groups=pd.Series(clst_assortment, name='ClusterId'))

PALETTE_18 = ['#808000', '#008000', '#00FF00', '#FF0000', '#800000', '#008080', '#0000FF', '#800080',
              '#000080', '#FF8C00', '#4B0082', '#48D1CC', '#DC143C', '#000000', '#808080',
              '#e41a1c', '#b35806', '#f781bf']
fig = plt.figure(figsize=(15, 9))
ax = sns.scatterplot(x='x', y='y', data=tsne_coords, hue='ID_groups', legend='full', palette=sns.color_palette(PALETTE_18))

### Сборка итоговой таблицы и визуализация по бренду, локации, формату выкладки

In [ ]:
result = assortment_mix_wide.copy().reset_index()
result = result.assign(ID_groups=pd.Series(clst_assortment, name='ClusterId'))
result = attach_trade_point_info(result.set_index('TradePointId'))

# добавляем среднюю стоимость покупки для контекста в визуализациях
cheques_and_revenue = df.groupby('TradePointId', as_index=False).agg({'Sum_Purch': 'sum', 'ChequeHeadersId': 'count'})
cheques_and_revenue.rename({'Sum_Purch': 'Revenue', 'ChequeHeadersId': 'ChequeCount'}, axis=1, inplace=True)
cheques_and_revenue = cheques_and_revenue.assign(mean_purch=lambda x: x.Revenue / x.ChequeCount)
cheques_and_revenue = cheques_and_revenue.merge(
    trade_points[['STORE_LOCATION_ID', 'TradePointCode']], left_on='TradePointId', right_on='STORE_LOCATION_ID', how='left'
).drop(['STORE_LOCATION_ID'], axis=1)

result = result.merge(cheques_and_revenue[['TradePointCode', 'Revenue', 'mean_purch']], how='left', on='TradePointCode')

# экспорт (не выполняется в портфолио-версии):
# result.to_excel("groups_ratio_clustering.xlsx", index=False)
# result.groupby('ID_groups').describe().to_excel("groups_ratio_clustering_stats.xlsx")
result

In [ ]:
PALETTE_11 = ['#0000FF', '#800000', '#00FF00', '#000080', '#FF0000', '#FF8C00',
              '#48D1CC', '#4B0082', '#DC143C', '#808080', '#FF00FF']

fig = plt.figure(figsize=(17, 10))
sns.swarmplot(x='ID_groups', y='Revenue', hue='BrandName', data=result, size=7, palette=PALETTE_11)
plt.show()

Дополнительный срез - по формату выкладки точки (`ОФТ` - открытая форма торговли, `ЗФТ` - закрытая) и по
географической локации (справочник локаций в портфолио не включён).

In [ ]:
trade_points_layout = pd.read_sql("""
    SELECT TradePointCode, Layout
    FROM dwh.TradePoints
    """, con=get_db_engine('sas_ia'))
trade_points_layout = trade_points_layout.replace({"Layout": {3.0: 'ОФТ', 2.0: 'ЗФТ'}})

result = result.merge(trade_points_layout, on='TradePointCode', how='left')

fig = plt.figure(figsize=(17, 10))
sns.swarmplot(x='ID_groups', y='Revenue', hue='Layout', data=result.query('Layout in ["ОФТ", "ЗФТ"]'), size=7)
plt.show()